# Timing GNN — Interactive Notebook

Loads the preprocessed `.pt` files from `preprocessed/`, trains `TimingGNN` (v4),
evaluates per-design AT R², and plots results.

**Paper split**: 15 train (DAC 2022 official 14 + alu4bit), 7 test (DAC 2022 fixed).  
**Calibration designs** (`uni_counter4b`, `cla8b`, `gcd`, `seq_detector`, `priority_enc`) are excluded from training — they are reserved for split-conformal calibration.

All code imports from `src/` — no duplication of model/feature logic here.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'GCNN', 'src'))
# If running the notebook from C:\Krish\GCNN\, just:
sys.path.insert(0, 'src')

import json, time, copy, random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display, clear_output

from model_configurable_v4 import ModelConfig, TimingGNN, nd_to_log, slew_to_log
from feature_sets import select_features, FEATURE_SETS

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

## Config
Edit the cells below to change the run. Everything else runs automatically.

In [ ]:
# ---- paths ----
PREPROCESSED_DIR = r'C:\Krish\GCNN\preprocessed'

# ---- design split (DO NOT MODIFY without reading CLAUDE.md §2 and §4) ----

# DAC 2022 official 7 test designs — FIXED, never train on these
PAPER_TEST = ['aes192', 'jpeg_encoder', 'spm', 'synth_ram', 'usbf_device', 'xtea', 'y_huff']

# Conformal calibration pool — FIXED, never train on these
# (training on calibration designs breaks split-conformal statistical validity)
CALIBRATION_DESIGNS = {'uni_counter4b', 'cla8b', 'gcd', 'seq_detector', 'priority_enc'}

# Training set: DAC 2022 official 14 + alu4bit (1 extra, not in calibration)
PAPER_TRAIN = [
    'BM64', 'aes128', 'aes256', 'aes_cipher', 'blabla',
    'cic_decimator', 'des', 'genericfir', 'picorv32a', 'salsa20',
    'usb', 'usb_cdc_core', 'wbqspiflash', 'zipdiv',  # DAC 2022 official 14
    'alu4bit',                                         # 1 extra
]

# Leakage guard — hard-fail if any train design is in test or calibration sets
_test_set = set(PAPER_TEST)
for d in PAPER_TRAIN:
    assert d not in _test_set, f"LEAKAGE: {d} is in PAPER_TEST"
    assert d not in CALIBRATION_DESIGNS, f"LEAKAGE: {d} is in CALIBRATION_DESIGNS"
print(f'Split check OK: {len(PAPER_TRAIN)} train | {len(PAPER_TEST)} test | {len(CALIBRATION_DESIGNS)} calibration (all disjoint)')

# ---- model hyperparams ----
HIDDEN_DIM  = 32
FEATURE_SET = 'no_spef16'   # 'full18' | 'no_spef16' | 'paper10'

# ---- training hyperparams ----
EPOCHS      = 500
LR          = 1e-3
PATIENCE    = 50
LOG_EVERY   = 10
SEED        = 1

print(f'Feature set : {FEATURE_SET} -> {len(FEATURE_SETS[FEATURE_SET])} features')
print(f'Train designs ({len(PAPER_TRAIN)}): {PAPER_TRAIN}')
print(f'Test  designs ({len(PAPER_TEST)}):  {PAPER_TEST}')

## Load preprocessed data

In [ ]:
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

with open(os.path.join(PREPROCESSED_DIR, 'meta.json')) as f:
    meta = json.load(f)
gate_vocab = meta['gate_vocab']
n_gates    = len(gate_vocab)
feat_names = meta.get('node_feature_names', [f'f{i}' for i in range(meta['NODE_FEATURES'])])
print(f'Gate vocab: {n_gates} types')
print(f'Node features ({meta["NODE_FEATURES"]}): {feat_names}')

# index all .pt files
pt_files = {
    os.path.splitext(fn)[0].replace('.graph', ''): os.path.join(PREPROCESSED_DIR, fn)
    for fn in os.listdir(PREPROCESSED_DIR) if fn.endswith('.pt')
}
print(f'\nFound {len(pt_files)} .pt files: {sorted(pt_files)}')

In [ ]:
KEEP_CPU = {'levels', 'net_by_level', 'cell_by_level'}

def move_to_device(g, device):
    return {k: (v if k in KEEP_CPU or not torch.is_tensor(v) else v.to(device))
            for k, v in g.items()}

def load_pt(path, gate_vocab):
    g = torch.load(path, map_location='cpu', weights_only=False)
    n_types = len(gate_vocab)
    unk = gate_vocab.get('UNKNOWN', 0)
    gts = g.get('cell_gtype_str', [])
    if gts:
        oh = torch.zeros(len(gts), n_types)
        for i, gt in enumerate(gts):
            oh[i, gate_vocab.get(gt, unk)] = 1.0
        g['gate_onehot'] = oh
    if 'gate_onehot' in g and g['gate_onehot'].shape[1] != n_types:
        old = g['gate_onehot']
        new = torch.zeros(old.shape[0], n_types)
        new[:, :old.shape[1]] = old
        g['gate_onehot'] = new
    # expand single-corner LUTs to 8 corners if needed
    if 'lut_delay' in g and g['lut_delay'].dim() == 3:
        g['lut_delay'] = g['lut_delay'].unsqueeze(1).expand(-1, 8, -1, -1).contiguous()
        g['lut_slew']  = g['lut_slew'].unsqueeze(1).expand(-1, 8, -1, -1).contiguous()
    return g


def load_split(names, feature_set):
    graphs = {}
    n_feat = None
    missing = []
    for n in names:
        if n not in pt_files:
            missing.append(n); continue
        g = load_pt(pt_files[n], gate_vocab)
        g, nf = select_features(g, feature_set)
        graphs[n] = g
        n_feat = nf
    if missing:
        print(f'  WARNING: not found in preprocessed: {missing}')
    return graphs, n_feat


print('Loading train graphs...')
train_graphs, n_feat = load_split(PAPER_TRAIN, FEATURE_SET)
print(f'  loaded {len(train_graphs)} train graphs | {n_feat} features')

print('Loading test graphs...')
test_graphs, _ = load_split(PAPER_TEST, FEATURE_SET)
print(f'  loaded {len(test_graphs)} test graphs')

## Inspect a design's tensor structure

In [ ]:
def inspect_graph(g, name):
    N = g['x'].shape[0]
    E_net  = g['net_src'].shape[0]
    E_cell = g['cell_src'].shape[0]
    n_ep   = g['endpoints'].shape[0]
    n_lvl  = len(g['levels'])
    print(f'=== {name} ===')
    print(f'  Nodes         : {N:,}')
    print(f'  Net edges     : {E_net:,}')
    print(f'  Cell edges    : {E_cell:,}')
    print(f'  Endpoints     : {n_ep:,}')
    print(f'  Topo levels   : {n_lvl}')
    print(f'  x shape       : {tuple(g["x"].shape)}')
    print(f'  gate_onehot   : {tuple(g["gate_onehot"].shape)}')
    print(f'  lut_delay     : {tuple(g["lut_delay"].shape)}')
    if 'AT_true' in g:
        at = g['AT_true']
        print(f'  AT_true       : min={at.min():.3f}  max={at.max():.3f}  mean={at.mean():.3f} ns')
    if 'slack_true' in g:
        sl = g['slack_true'][g['endpoints']]
        print(f'  slack_true[ep]: min={sl.min():.3f}  max={sl.max():.3f}  mean={sl.mean():.3f} ns')
    masks = ['slew_real_mask', 'net_trusted_mask', 'cell_lut_mask']
    for m in masks:
        if m in g:
            print(f'  {m:20}: {g[m].sum().item():,} / {g[m].numel():,} ({100*g[m].float().mean():.1f}%)')

# inspect one train + one test design
sample_train = next(iter(train_graphs))
sample_test  = next(iter(test_graphs))
inspect_graph(train_graphs[sample_train], sample_train)
print()
inspect_graph(test_graphs[sample_test], sample_test)

## Model setup

In [ ]:
cfg   = ModelConfig(hidden_dim=HIDDEN_DIM)
model = TimingGNN(cfg, n_gates=n_gates, in_dim=n_feat).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {cfg.describe()}')
print(f'Params: {n_params:,}')
print(f'Device: {DEVICE}')

## Training

In [ ]:
def r2(pred, true):
    sst = ((true - true.mean()) ** 2).sum()
    if sst < 1e-8: return float('nan')
    return float(1 - ((pred - true) ** 2).sum() / sst)


@torch.no_grad()
def eval_test():
    model.eval()
    pooled_p, pooled_t, per = [], [], {}
    for name, gc in test_graphs.items():
        g   = move_to_device(gc, DEVICE)
        out = model(g)
        ep  = g['endpoints']
        p   = out['AT'][ep].cpu()
        t   = g['AT_true'][ep].cpu()
        pooled_p.append(p); pooled_t.append(t)
        per[name] = r2(p, t)
    pooled = r2(torch.cat(pooled_p), torch.cat(pooled_t))
    mean_per = float(np.nanmean(list(per.values())))
    return pooled, mean_per, per


opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt, factor=0.6, patience=max(1, EPOCHS // 8), min_lr=1e-4)
mse   = nn.MSELoss()

epoch_log = []   # [(epoch, loss, mean_per_r2, pooled_r2, lr)]
best_r2, best_state, best_ep, bad = -1e9, None, -1, 0

print(f'Training {len(train_graphs)} designs for up to {EPOCHS} epochs...')
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    tl = dict(total=0, AT=0, slew=0, net=0, cell=0)

    for gc in train_graphs.values():
        g   = move_to_device(gc, DEVICE)
        out = model(g, groundtruth=True)
        ep  = g['endpoints']

        L_AT = mse(out['AT'][ep], g['AT_true'][ep])

        sm = g.get('slew_real_mask')
        L_slew = (mse(out['SLEW_log'][sm], slew_to_log(g['SLEW_true'][sm]))
                  if sm is not None and sm.sum() > 0 else torch.tensor(0.0, device=DEVICE))

        nm = g.get('net_trusted_mask')
        L_net = (mse(out['net_delay_log'][nm], nd_to_log(g['nd_true'][nm]))
                 if nm is not None and nm.sum() > 0 else torch.tensor(0.0, device=DEVICE))

        cm = g.get('cell_lut_mask')
        L_cell = (mse(out['cell_delay'][cm], g['cd_true'][cm])
                  if cm is not None and cm.sum() > 0 else torch.tensor(0.0, device=DEVICE))

        loss = L_AT + L_slew + L_net + L_cell
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        opt.step()
        tl['total'] += loss.item(); tl['AT'] += L_AT.item()
        tl['slew']  += L_slew.item(); tl['net'] += L_net.item()
        tl['cell']  += L_cell.item()

    n_tr = max(len(train_graphs), 1)
    sched.step(tl['total'] / n_tr)

    if epoch == 1 or epoch % LOG_EVERY == 0:
        pooled, mean_per, per = eval_test()
        lr_now = opt.param_groups[0]['lr']
        epoch_log.append((epoch, tl['total']/n_tr, mean_per, pooled, lr_now))
        elapsed = time.time() - t_start
        print(f'  ep {epoch:4d} | loss {tl["total"]/n_tr:7.4f} '
              f'[AT {tl["AT"]/n_tr:.4f} slew {tl["slew"]/n_tr:.4f} '
              f'net {tl["net"]/n_tr:.4f} cell {tl["cell"]/n_tr:.4f}] '
              f'| test r2 pooled={pooled:.4f}  mean={mean_per:.4f} '
              f'| lr {lr_now:.2e}  {elapsed:.0f}s')

        if mean_per > best_r2 + 1e-4:
            best_r2, best_ep, bad = mean_per, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= PATIENCE // LOG_EVERY + 1:
                print(f'  [early stop] restoring best weights from epoch {best_ep}')
                break

if best_state:
    model.load_state_dict(best_state)
print(f'\nDone. Best mean-per-design AT R² = {best_r2:.4f} at epoch {best_ep}')

## Training curve

In [ ]:
epochs_v = [r[0] for r in epoch_log]
loss_v   = [r[1] for r in epoch_log]
mean_r2  = [r[2] for r in epoch_log]
pool_r2  = [r[3] for r in epoch_log]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs_v, loss_v, color='steelblue', lw=1.5)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Train loss')
ax1.set_title('Training loss'); ax1.grid(alpha=0.3)

ax2.plot(epochs_v, mean_r2, label='mean-per-design', color='tab:orange', lw=1.5)
ax2.plot(epochs_v, pool_r2, label='pooled',          color='tab:blue',   lw=1.5, ls='--')
ax2.axvline(best_ep, color='red', ls=':', lw=1, label=f'best ep {best_ep}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('AT R²')
ax2.set_title('Test AT R² over training')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Per-design evaluation

In [ ]:
results = {}

model.eval()
with torch.no_grad():
    for name, gc in test_graphs.items():
        g   = move_to_device(gc, DEVICE)
        t0  = time.time()
        out = model(g)
        dt  = time.time() - t0
        ep  = g['endpoints']

        at_pred = out['AT'][ep].cpu()
        at_true = g['AT_true'][ep].cpu()
        at_r2   = r2(at_pred, at_true)

        has_slack = 'slack_true' in g
        sl_r2 = float('nan')
        if has_slack:
            sl_t = g['slack_true'][ep].cpu()
            sl_p = sl_t + at_true - at_pred      # RAT - AT_pred
            sl_r2 = r2(sl_p, sl_t)

        nd_r2 = float('nan')
        if 'nd_true' in g and g['nd_true'].numel() > 0:
            nm = g.get('net_trusted_mask')
            nd_p = out['net_delay_log'].cpu() if 'net_delay_log' in out else out.get('net_delay', torch.zeros(1)).cpu()
            nd_t = nd_to_log(g['nd_true'].cpu()) if 'net_delay_log' in out else g['nd_true'].cpu()
            if nm is not None:
                nm_cpu = nm.cpu()
                if nm_cpu.sum() > 0:
                    nd_r2 = r2(nd_p[nm_cpu], nd_t[nm_cpu])
            else:
                nd_r2 = r2(nd_p, nd_t)

        cd_r2 = float('nan')
        if 'cd_true' in g and g['cd_true'].numel() > 0:
            cm = g.get('cell_lut_mask')
            cd_p = out['cell_delay'].cpu()
            cd_t = g['cd_true'].cpu()
            if cm is not None:
                cm_cpu = cm.cpu()
                if cm_cpu.sum() > 0:
                    cd_r2 = r2(cd_p[cm_cpu], cd_t[cm_cpu])
            else:
                cd_r2 = r2(cd_p, cd_t)

        results[name] = dict(at_r2=at_r2, sl_r2=sl_r2, nd_r2=nd_r2, cd_r2=cd_r2,
                              at_pred=at_pred.numpy(), at_true=at_true.numpy(),
                              infer_s=dt)
        print(f'  {name:15} AT R²={at_r2:.4f}  slack R²={sl_r2:.4f}  '
              f'net R²={nd_r2:.4f}  cell R²={cd_r2:.4f}  ({dt:.2f}s)')

pooled_p = np.concatenate([r['at_pred'] for r in results.values()])
pooled_t = np.concatenate([r['at_true'] for r in results.values()])
print(f'\nPooled AT R² = {r2(torch.tensor(pooled_p), torch.tensor(pooled_t)):.4f}')
print(f'Mean-per-design AT R² = {np.nanmean([r["at_r2"] for r in results.values()]):.4f}')

## Scatter plots — AT prediction vs ground truth

In [ ]:
n = len(results)
ncols = 4
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4 * nrows))
axes = axes.flatten()

for ax, (name, res) in zip(axes, results.items()):
    at_t = res['at_true']
    at_p = res['at_pred']
    lo, hi = min(at_t.min(), at_p.min()), max(at_t.max(), at_p.max())
    ax.scatter(at_t, at_p, s=1, alpha=0.25, color='steelblue', rasterized=True)
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1, label='ideal')
    ax.set_xlabel('AT true (ns)')
    ax.set_ylabel('AT pred (ns)')
    ax.set_title(f'{name}\nAT R²={res["at_r2"]:.4f}')
    ax.grid(alpha=0.2)

for ax in axes[len(results):]:
    ax.set_visible(False)

plt.suptitle('Endpoint Arrival Time: prediction vs ground truth', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## Per-design R² bar chart

In [ ]:
names = list(results.keys())
at_r2s = [results[n]['at_r2'] for n in names]
sl_r2s = [results[n]['sl_r2'] for n in names]

x = np.arange(len(names))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 4))
bars1 = ax.bar(x - w/2, at_r2s, w, label='AT R²',    color='steelblue')
bars2 = ax.bar(x + w/2, sl_r2s, w, label='Slack R²', color='coral', alpha=0.8)

ax.axhline(np.nanmean(at_r2s), color='steelblue', ls='--', lw=1,
           label=f'mean AT R²={np.nanmean(at_r2s):.3f}')
ax.set_xticks(x); ax.set_xticklabels(names, rotation=30, ha='right')
ax.set_ylabel('R²')
ax.set_ylim(min(0, min(at_r2s) - 0.05), 1.02)
ax.set_title('Per-design AT and Slack R²')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Save model checkpoint

In [ ]:
import datetime

ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
ckpt_dir = os.path.join('src', 'results', f'nb_{ts}_train{len(train_graphs)}_test{len(test_graphs)}')
os.makedirs(ckpt_dir, exist_ok=True)

# model checkpoint
torch.save({'model_state': model.state_dict(),
            'cfg': cfg, 'n_feat': n_feat, 'n_gates': n_gates,
            'best_epoch': best_ep, 'best_r2': best_r2,
            'feature_set': FEATURE_SET},
           os.path.join(ckpt_dir, 'model_checkpoint.pt'))

# epoch log CSV
import csv
with open(os.path.join(ckpt_dir, 'epoch_log.csv'), 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['epoch', 'loss', 'mean_per_design_AT_R2', 'pooled_AT_R2', 'lr'])
    w.writerows(epoch_log)

# per-design results JSON
summary = {n: {k: float(v) for k, v in res.items() if k not in ('at_pred', 'at_true')}
           for n, res in results.items()}
summary['_pooled_AT_R2'] = float(r2(torch.tensor(pooled_p), torch.tensor(pooled_t)))
summary['_mean_per_design_AT_R2'] = float(np.nanmean([r['at_r2'] for r in results.values()]))
with open(os.path.join(ckpt_dir, 'results.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Saved to: {ckpt_dir}')
print(json.dumps(summary, indent=2))

## (Optional) Conformal slack intervals

Requires having calibration designs (held-out from test).  
Uses `conformal_v4.py` from `src/`.

In [ ]:
# Export per-design prediction CSVs for conformal.py
csv_dir = os.path.join(ckpt_dir, 'csv')
os.makedirs(csv_dir, exist_ok=True)

model.eval()
with torch.no_grad():
    for name, gc in test_graphs.items():
        g   = move_to_device(gc, DEVICE)
        out = model(g)
        ep  = g['endpoints']
        N   = g['x'].shape[0]
        ep_mask = torch.zeros(N, dtype=torch.bool)
        ep_mask[ep] = True

        at_t = g['AT_true'].cpu()
        at_p = out['AT'].cpu()
        has_slack = 'slack_true' in g
        if has_slack:
            sl_t = g['slack_true'].cpu()
            sl_p = sl_t + at_t - at_p

        with open(os.path.join(csv_dir, f'predictions_{name}.csv'), 'w', newline='') as f:
            w = csv.writer(f)
            hdr = ['node', 'is_endpoint', 'at_true_0', 'at_pred_0']
            if has_slack:
                hdr += ['slack_true', 'slack_pred']
            w.writerow(hdr)
            for i in range(N):
                row = [i, int(ep_mask[i]), float(at_t[i]), float(at_p[i])]
                if has_slack:
                    row += [float(sl_t[i]), float(sl_p[i])]
                w.writerow(row)

print(f'Prediction CSVs written to: {csv_dir}')
print(f'To run conformal (split calibration on usbf_device + y_huff, test remaining 5):')
print(f'  python src/conformal_v4.py --csv_dir "{csv_dir}" \\')
print(f'      --cal_designs usbf_device,y_huff \\')
print(f'      --test_designs xtea,spm,synth_ram,aes192,jpeg_encoder \\')
print(f'      --target slack --alpha 0.1')